# 02 · HTDemucs v4 — Separación de fuentes

Notebook de inferencia con HTDemucs v4 (paquete pip `demucs`, API de alto nivel `demucs.api.Separator`).
Categoría: **separación de fuentes** (voz / batería / bajo / resto). Se compara contra el baseline
clásico HPSS (Harmonic-Percussive Source Separation).

Requiere haber ejecutado antes `00_Setup_Base.ipynb` (Drive montado, HF_HOME configurado, utils
guardadas en `utils/`, audio del tutor subido a `audio_samples/`).

**A diferencia de DeepFilterNet, `demucs` es un paquete pip puro** (sin dependencias nativas en
Rust/C que haya que compilar), así que no hace falta instalar ningún compilador ni reiniciar el
kernel a mitad de notebook.


## 1. Montar Drive y configurar entorno

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/Proyecto_Audio'
os.environ['HF_HOME'] = f'{PROJECT_ROOT}/cache'
os.environ['HF_HUB_CACHE'] = f'{PROJECT_ROOT}/cache'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Instalar dependencias comunes

Los paquetes de `pip` no persisten entre sesiones de Colab (solo los archivos en Drive sí), así
que hay que reinstalar estas dependencias en cada sesión nueva.


In [5]:
!pip install -q librosa soundfile scipy pesq pystoi speechmos matplotlib pandas onnxruntime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 64.8 MB/s eta 0:00:00


## 3. Actualizar y cargar utilidades comunes (desde utils/ en Drive)

Añadimos al módulo de baselines clásicos una función nueva, `baseline_separacion_hpss`, que no
existía en el `00_Setup_Base` original (los baselines de ese notebook cubrían denoising, dereverb,
BWE y de-clipping, pero no separación de fuentes). Se guarda esta versión ampliada en Drive para
que quede disponible en futuros notebooks igual que el resto de utilidades.


In [6]:
%%writefile /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_baselines_clasicos.py

import numpy as np
import librosa
import scipy.signal as signal
from scipy.interpolate import CubicSpline


def baseline_denoising_spectral_gating(audio, sr, umbral_db=-20):
    """
    Baseline de denoising: 'spectral gating' clásico. Estima el ruido de fondo a partir de los
    frames más silenciosos y resta ese perfil espectral del resto de la señal.
    """
    stft = librosa.stft(audio)
    magnitud, fase = np.abs(stft), np.angle(stft)

    perfil_ruido = np.percentile(magnitud, 10, axis=1, keepdims=True)

    umbral = perfil_ruido * (10 ** (umbral_db / 20))
    magnitud_limpia = np.where(magnitud > umbral, magnitud - perfil_ruido, 0.0)
    magnitud_limpia = np.maximum(magnitud_limpia, 0.0)

    stft_limpio = magnitud_limpia * np.exp(1j * fase)
    return librosa.istft(stft_limpio, length=len(audio))


def baseline_dereverb_filtro_paso_alto(audio, sr, frecuencia_corte=100):
    """
    Baseline de dereverberation: filtro paso-alto simple.
    """
    sos = signal.butter(4, frecuencia_corte, btype='high', fs=sr, output='sos')
    return signal.sosfilt(sos, audio)


def baseline_bwe_interpolacion_spline(audio, sr_origen, sr_destino):
    """
    Baseline de super-resolución/BWE: upsampling clásico por interpolación spline cúbica.
    """
    return librosa.resample(audio, orig_sr=sr_origen, target_sr=sr_destino, res_type='fft')


def baseline_declipping_interpolacion_cubica(audio, umbral=0.99):
    """
    Baseline de de-clipping: interpolación cúbica sobre muestras saturadas.
    """
    audio_reparado = audio.copy()
    indices_clipeados = np.where(np.abs(audio) >= umbral)[0]

    if len(indices_clipeados) == 0:
        return audio_reparado

    indices_validos = np.where(np.abs(audio) < umbral)[0]
    if len(indices_validos) < 4:
        return audio_reparado

    spline = CubicSpline(indices_validos, audio[indices_validos])
    audio_reparado[indices_clipeados] = spline(indices_clipeados)
    return audio_reparado


def baseline_separacion_hpss(audio, sr):
    """
    Baseline de separación de fuentes: HPSS (Harmonic-Percussive Source Separation) clásico
    de librosa, vía median-filtering en el espectrograma (Fitzgerald, 2010).

    No es un separador de 4 stems como HTDemucs (voz/batería/bajo/resto): es un método clásico
    de procesado de señal que solo distingue entre componente armónico (más parecido a voz e
    instrumentos melódicos) y componente percusivo (más parecido a batería/ataques transitorios).
    Se usa aquí como referencia no-IA más cercana conceptualmente a la separación de fuentes,
    comparando su componente armónico contra el stem 'vocals' de HTDemucs.

    Returns:
        armonico (np.ndarray): componente armónico (proxy clásico de "voz/melodía").
        percusivo (np.ndarray): componente percusivo (proxy clásico de "batería/ritmo").
    """
    armonico, percusivo = librosa.effects.hpss(audio)
    return armonico, percusivo


Overwriting /content/drive/MyDrive/Proyecto_Audio/utils/audio_utils_baselines_clasicos.py


In [7]:
import sys
sys.path.append(f'{PROJECT_ROOT}/utils')

from audio_utils_funcionescomunes import cargar_audio, guardar_audio, resamplear, normalizar_pico
from audio_utils_memoria_GPU import liberar_memoria_gpu
from audio_utils_metricas_no_intrusivas import calcular_dnsmos
from audio_utils_baselines_clasicos import baseline_separacion_hpss


## 4. Instalar dependencias específicas de HTDemucs

`demucs` es un paquete pip normal (sin extensiones nativas que compilar), así que la instalación
es directa. Trae su propia copia de PyTorch como dependencia si hiciera falta, pero en Colab ya
hay una versión compatible preinstalada, así que normalmente reutiliza esa.


In [8]:
!pip install -q demucs


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 32.2 MB/s eta 0:00:00


## 5. Cargar el audio de prueba

Muestra los audios disponibles en `audio_samples/` y pide cuál usar.


In [9]:
carpeta_audios = f'{PROJECT_ROOT}/audio_samples'

print('Audios disponibles en audio_samples/:')
for archivo in os.listdir(carpeta_audios):
    print(f'  - {archivo}')

nombre_audio = input('\nIntroduce el nombre del archivo de audio a usar (con extensión, ej. AUDIO_TFG.wav): ').strip()
RUTA_AUDIO_ORIGINAL = f'{carpeta_audios}/{nombre_audio}'

if not os.path.isfile(RUTA_AUDIO_ORIGINAL):
    raise FileNotFoundError(f'No se ha encontrado el archivo: {RUTA_AUDIO_ORIGINAL}')

# Nombre base sin extensión, para usarlo luego al nombrar los archivos de salida
NOMBRE_BASE = os.path.splitext(nombre_audio)[0]

# Cargamos con su sample rate original para el baseline HPSS; HTDemucs resamplea internamente
audio_original, sr_original = cargar_audio(RUTA_AUDIO_ORIGINAL, sr_objetivo=None, forzar_mono=True)
print(f'\nAudio cargado: {len(audio_original)/sr_original:.1f} s, {sr_original} Hz')


Audios disponibles en audio_samples/:
  - AUDIO_REVERB_ALBIOL_TFG.wav

Introduce el nombre del archivo de audio a usar (con extensión, ej. AUDIO_TFG.wav): AUDIO_REVERB_ALBIOL_TFG.wav

Audio cargado: 88.0 s, 44100 Hz


## 6. Cargar el modelo HTDemucs v4

`Separator` es la API de alto nivel de `demucs` (desde la v4): dado un modelo preentrenado
(`htdemucs` por defecto), se encarga de resamplear a 44.1 kHz, convertir a estéreo si hace falta,
trocear el audio en ventanas con solape para no saturar la GPU, y recombinar los 4 stems al final.
No hace falta gestionar nada de eso a mano.

Stems que separa: `drums` (batería), `bass` (bajo), `other` (resto de instrumentación) y `vocals`
(voz) — este último es el más relevante para un proyecto centrado en restauración de audio de voz.


In [10]:
from demucs.api import Separator

separador = Separator(model='htdemucs')
print('Modelo HTDemucs v4 cargado. Stems disponibles:', separador._model.sources)


htdemucs.yaml:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

955717e8.safetensors:   0%|          | 0.00/84.0M [00:00<?, ?B/s]

Modelo HTDemucs v4 cargado. Stems disponibles: ['drums', 'bass', 'other', 'vocals']


## 7. Inferencia: separación de fuentes

In [11]:
origen, stems = separador.separate_audio_file(RUTA_AUDIO_ORIGINAL)

carpeta_salida_htdemucs = f'{PROJECT_ROOT}/outputs/htdemucs/{NOMBRE_BASE}'
os.makedirs(carpeta_salida_htdemucs, exist_ok=True)

for nombre_stem, audio_stem in stems.items():
    ruta_stem = f'{carpeta_salida_htdemucs}/{nombre_stem}.wav'
    # audio_stem viene como tensor (canales, muestras) a 44100 Hz; guardar_audio/soundfile
    # espera (muestras, canales) o mono, así que transponemos antes de guardar.
    guardar_audio(ruta_stem, audio_stem.numpy().T, 44100)

print(f'\nStems guardados en: {carpeta_salida_htdemucs}')

# Nos quedamos con el stem de voz como referencia principal para las métricas de este notebook
audio_vocals = stems['vocals'].numpy().mean(axis=0)  # a mono para comparar con el resto


Audio guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/htdemucs/AUDIO_REVERB_ALBIOL_TFG/drums.wav
Audio guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/htdemucs/AUDIO_REVERB_ALBIOL_TFG/bass.wav
Audio guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/htdemucs/AUDIO_REVERB_ALBIOL_TFG/other.wav
Audio guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/htdemucs/AUDIO_REVERB_ALBIOL_TFG/vocals.wav

Stems guardados en: /content/drive/MyDrive/Proyecto_Audio/outputs/htdemucs/AUDIO_REVERB_ALBIOL_TFG


## 8. Baseline clásico (HPSS) sobre el mismo audio

HPSS no separa 4 stems como HTDemucs, solo distingue componente armónico (proxy clásico de
voz/melodía) de componente percusivo (proxy clásico de batería/ritmo). Se compara su componente
armónico contra el stem `vocals` de HTDemucs, como referencia no-IA más cercana conceptualmente.


In [12]:
audio_armonico, audio_percusivo = baseline_separacion_hpss(audio_original, sr_original)

carpeta_salida_baseline = f'{PROJECT_ROOT}/outputs/baseline_separacion/{NOMBRE_BASE}'
os.makedirs(carpeta_salida_baseline, exist_ok=True)

guardar_audio(f'{carpeta_salida_baseline}/armonico.wav', audio_armonico, sr_original)
guardar_audio(f'{carpeta_salida_baseline}/percusivo.wav', audio_percusivo, sr_original)

print(f'Baseline HPSS guardado en: {carpeta_salida_baseline}')


Audio guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/baseline_separacion/AUDIO_REVERB_ALBIOL_TFG/armonico.wav
Audio guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/baseline_separacion/AUDIO_REVERB_ALBIOL_TFG/percusivo.wav
Baseline HPSS guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/baseline_separacion/AUDIO_REVERB_ALBIOL_TFG


## 9. Calcular métricas (DNSMOS)

Como no hay audio limpio de referencia para la grabación del tutor, usamos DNSMOS (métrica
no-intrusiva de calidad de voz) sobre: mezcla original, stem `vocals` de HTDemucs, y componente
armónico del baseline HPSS. `calcular_dnsmos` resamplea a 16 kHz y reduce a mono automáticamente.

**Nota:** DNSMOS está pensado para evaluar calidad de voz, no separación multi-instrumento en
general, así que aquí solo es un proxy de qué tan "limpia y tipo voz" suena la fuente aislada por
cada método — no mide directamente si el contenido separado es correcto (para eso harían falta
métricas con referencia tipo SI-SDR, que no aplican sin stems de verdad limpios de este audio).


In [13]:
dnsmos_original = calcular_dnsmos(audio_original, sr_original)
dnsmos_vocals_ia = calcular_dnsmos(audio_vocals, 44100)
dnsmos_armonico_baseline = calcular_dnsmos(audio_armonico, sr_original)

print('DNSMOS — Mezcla original:            ', dnsmos_original)
print('DNSMOS — HTDemucs (stem vocals, IA): ', dnsmos_vocals_ia)
print('DNSMOS — Baseline HPSS (armónico):   ', dnsmos_armonico_baseline)


DNSMOS — Mezcla original:             {'ovrl_mos': np.float64(1.3853487473266228), 'sig_mos': np.float64(1.5779884955663657), 'bak_mos': np.float64(1.790054065636629), 'p808_mos': np.float32(2.426125)}
DNSMOS — HTDemucs (stem vocals, IA):  {'ovrl_mos': np.float64(1.4014474715568774), 'sig_mos': np.float64(1.5942317983489362), 'bak_mos': np.float64(1.8418853918724127), 'p808_mos': np.float32(2.4327567)}
DNSMOS — Baseline HPSS (armónico):    {'ovrl_mos': np.float64(1.6423270633392273), 'sig_mos': np.float64(1.8024033880085697), 'bak_mos': np.float64(2.2652256110414695), 'p808_mos': np.float32(2.4748776)}


## 10. Tabla resumen de la comparativa

In [14]:
import pandas as pd

resumen = pd.DataFrame([
    {'Version': 'Mezcla original',        'OVRL': dnsmos_original.get('ovrl_mos'),        'SIG': dnsmos_original.get('sig_mos'),        'BAK': dnsmos_original.get('bak_mos')},
    {'Version': 'HTDemucs vocals (IA)',    'OVRL': dnsmos_vocals_ia.get('ovrl_mos'),       'SIG': dnsmos_vocals_ia.get('sig_mos'),       'BAK': dnsmos_vocals_ia.get('bak_mos')},
    {'Version': 'Baseline HPSS (no-IA)',   'OVRL': dnsmos_armonico_baseline.get('ovrl_mos'), 'SIG': dnsmos_armonico_baseline.get('sig_mos'), 'BAK': dnsmos_armonico_baseline.get('bak_mos')},
])
resumen


,Version,OVRL,SIG,BAK
0,Mezcla original,1.385349,1.577988,1.790054
1,HTDemucs vocals (IA),1.401447,1.594232,1.841885
2,Baseline HPSS (no-IA),1.642327,1.802403,2.265226


## 11. Liberar memoria GPU

In [15]:
liberar_memoria_gpu(separador._model, 'separador')


Memoria GPU liberada. Uso actual: 0.01 GB


## Próximo paso

Con HTDemucs ya probado y comparado contra su baseline HPSS, el siguiente notebook sería el de
MP-SENet o ClearVoice/MossFormer2 (según el orden que fijemos), siguiendo la Fase 2 del proyecto.
